# 02 Hydrogen evolution (IA-HER-001)

**Requirement:** `IA-HER-001`

Patents: US12308414B2, EP4602674A1. Equations are paraphrased; see claims/paragraphs cited below.

## Governing equations


Parasitic charge reaction (EP [0044], [0047]):

$$2\,\mathrm{H_2O} + 2\,\mathrm{e^-} \to \mathrm{H_2} + 2\,\mathrm{OH^-}$$

Coulombic efficiency is

$$\mathrm{CE} = \frac{I_\mathrm{Fe}}{I_\mathrm{Fe} + I_\mathrm{HER}}$$

and falls when the ionic path to the back of a thick iron electrode exceeds
front-surface HER. Vertical channels are a bubble-egress path ([0056]).


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

here = Path.cwd()
for cand in (here, here.parent):
    if (cand / "src" / "ironair").is_dir() and (cand / "plant_sim").is_dir():
        sys.path.insert(0, str(cand))
        sys.path.insert(0, str(cand / "src"))
        break

from plant_sim.bootstrap import setup_path
setup_path()
from plant_sim.params import default_params, CAPACITY_FE_OH2_MAH_G, CAPACITY_MAGNETITE_MAH_G
from plant_sim.plotting import apply_style, timeseries, xy_plot, FIGURES
apply_style()
P = default_params()
print("6 M KOH default", P.c_KOH_mol_m3/1000, "M; T_ep", P.T_ep_sim_K, "K")
print("Faraday 960/320 checks", round(CAPACITY_FE_OH2_MAH_G,2), round(CAPACITY_MAGNETITE_MAH_G,2))
print("figures ->", FIGURES)


## Isolated component simulation

In [ ]:
from plant_sim.components.her import HydrogenEvolution
from plant_sim.components.base import integrate_component
her = HydrogenEvolution(P)
I = -80 * P.I_discharge_100h_A
t, y = integrate_component(lambda t,y: her.rhs(t,y,{"I_fe_A": I, "T_K": P.T_ep_sim_K, "L_path_m": P.L_anode_m, "a_oh": 6.0, "a_h2o": 0.72}, {}), her.y0(), (0, 1500), n_eval=180)
ce = [her.outputs(float(ti), y[:,i], {"I_fe_A": I, "T_K": P.T_ep_sim_K, "L_path_m": P.L_anode_m, "a_oh": 6.0, "a_h2o": 0.72}, {})["CE"] for i,ti in enumerate(t)]
timeseries(t/60, {"n_H2": y[0], "CE": ce, "theta": y[1]}, xlabel="t (min)", ylabel="n_H2 / CE / holdup", title="HER gas and coulombic efficiency", callout="EP [0044]-[0047]", stem="nb02_her")


## Verification against patents / SSS002

In [ ]:
print('Open-circuit HER is smaller than strong cathodic charging HER (IA-HER-001 comment).')